In [ ]:
import sys
import os

from pathlib import Path
from timeit import time

build = Path(os.getcwd()).parent / "build"
print("Buscando librerias en", build)

: 

In [2]:
sys.path.append(str(build))

from amd_ml_py import linear_regression

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

In [4]:
def create_matrix(n_points: int, n_dimentions: int) -> list[np.ndarray]:
    #We generate a linear regression with coefficients 1, 2, 3, ...
    #We count with the bias term

    terms = np.zeros(n_dimentions + 1, dtype=np.float32)
    terms[0] = 1

    for i in range(n_dimentions):
        terms[i+1] = i+2

    X = np.ones((n_points, 1), dtype = np.float32)

    for i in range(n_dimentions):
        X = np.hstack((
            X, 
            np.linspace(0, 1, n_points).reshape(-1, 1) + np.random.randn(n_points).reshape(-1, 1)
            ))
    
    y = np.matmul(X, terms.reshape(-1, 1))

    return [X, y, terms]

In [5]:
def comparation(X: np.ndarray, y: np.ndarray, terms: np.ndarray, print_data = True) -> list[float]:
    a = time.time()
    parameters_sk = LinearRegression(fit_intercept=False).fit(X, y)
    b = time.time()

    c = time.time()
    params: np.ndarray = linear_regression(
        X, y,
        n_iter=1000,
        tolerance=0.01,
        learning_rate=0.1
    )
    d = time.time()

    time_amd = d-c
    time_sk = b-a

    if print_data:
        print(f"Parámetros encontrados: {params.round(1)}, tiempo usado: {time_amd} milisegundos")
        print(f"Parámetros encontrados sklearn: {parameters_sk.coef_.round(1)}, tiempo usado: {time_sk} milisegundos")
        print(f"Esperado: {terms}")

    return [time_amd, time_sk] 

In [ ]:
from timeit import time

In [50]:
def xd(points, dimension):
    X, y, terms = create_matrix(points, dimension)

    a = time.time()
    params: np.ndarray = linear_regression(
        X, y.ravel()
    )
    b = time.time()
    #print("Tiempo amd -> ", b-a)

    c = time.time()
    LinearRegression(fit_intercept=False).fit(X, y.ravel())
    d = time.time()

    #print("Tiempo sk -> ", d-c)

    diferencia = b-a-d+c
    #print("Diferencia de tiempos -> ", diferencia)

    return diferencia

In [51]:
aux = np.inf
aux2 = 0

for i in [10**w for w in range(1, 6)]:
    for j in [e for e in range(2, 5)]:
        if (aux2 := xd(i, j)) < aux:
            aux = aux2
            print("Nuevo mejor tiempo -> ", aux, "Con puntos -> ", i, " y parametros -> ", j)

Nuevo mejor tiempo ->  0.09240460395812988 Con puntos ->  10  y parametros ->  2
Nuevo mejor tiempo ->  0.044385671615600586 Con puntos ->  100  y parametros ->  2
Nuevo mejor tiempo ->  -0.046061038970947266 Con puntos ->  1000  y parametros ->  2
Nuevo mejor tiempo ->  -0.14209342002868652 Con puntos ->  1000  y parametros ->  3
Nuevo mejor tiempo ->  -0.1889936923980713 Con puntos ->  10000  y parametros ->  3
Nuevo mejor tiempo ->  -0.3405418395996094 Con puntos ->  10000  y parametros ->  4
Nuevo mejor tiempo ->  -0.4071178436279297 Con puntos ->  100000  y parametros ->  2


In [ ]:
valores = np.zeros(2, 5)

for i in [10**w for w in range(1, 6)]:

    X, y, val = create_matrix(i, 2)

    a = time.time()
    params: np.ndarray = linear_regression(
        X, y.ravel()
    )
    b = time.time()

    c = time.time()
    LinearRegression(fit_intercept=False).fit(X, y.ravel())
    d = time.time()

    valores[0, i-1] = b-a
    valores[1, i-1] = d-c